# PokeTraining: recorrido reproducible

Este notebook muestra cómo configurar y ejecutar el agente DQN modular para
Pokémon Red. El entrenamiento está desactivado por defecto: ejecutar todas las
celdas no abre una ROM ni inicia una tarea costosa.

**Mantenedor:** Ignacio Cordova. El trabajo académico original acredita también
a Joaquín López y Vicente Silva. La versión histórica, con sus salidas intactas,
se conserva en [`archive/PokeTraining_original.ipynb`](archive/PokeTraining_original.ipynb).

## 1. Preparación

Instala el proyecto desde la raíz del repositorio:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements-dev.txt
python -m pip install -e . --no-deps
```

Coloca una ROM obtenida legalmente en `data/roms/POKEMON_RED.gb`. La ROM,
partidas, pesos y ejecuciones nuevas están excluidos de Git.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from poketraining.config import TrainerConfig
from poketraining.emulator import PokemonRedEnvironment
from poketraining.model import create_synchronized_models
from poketraining.trainer import PokemonAITrainer

candidates = (Path.cwd(), Path.cwd().parent)
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook desde la raíz del repositorio o notebooks/.")

ROM_PATH = PROJECT_ROOT / "data" / "roms" / "POKEMON_RED.gb"
RUN_TRAINING = False

## 2. Configuración verificada

Los valores siguientes conservan la arquitectura y los hiperparámetros centrales
del prototipo. Se usa un buffer pequeño en este ejemplo para evitar reservar más
de 1 GiB; los perfiles completos están en `configs/`.

In [ ]:
config = TrainerConfig(
    episodes=1,
    max_steps=100,
    replay_capacity=256,
    batch_size=32,
    screenshot_interval=0,
    checkpoint_every=0,
    reward_mode="exploration",
    seed=42,
)
config.to_dict()

## 3. Red DQN

La fábrica construye las redes en línea y objetivo y sincroniza sus pesos antes
del primer paso de optimización. Esta celda no necesita la ROM.

In [ ]:
online_model, target_model = create_synchronized_models(
    num_actions=6,
    observation_shape=config.observation_shape,
)
online_model.summary()

## 4. Validación local de la ROM

La revisión internacional admitida se valida primero mediante SHA-1. El título
del cartucho se comprueba después, al iniciar PyBoy. Solo se muestran huellas;
el archivo nunca se copia a los resultados.

In [ ]:
if ROM_PATH.is_file():
    environment = PokemonRedEnvironment(ROM_PATH, config)
    fingerprints = environment.validate_rom()
    print("ROM compatible; SHA-1:", fingerprints["sha1"])
else:
    print("ROM no encontrada. Sigue data/README.md antes de entrenar.")

## 5. Entrenamiento opcional

Cambia `RUN_TRAINING` a `True` únicamente después de revisar la configuración.
Cada ejecución usa un directorio nuevo y se detiene si ese nombre ya existe.

In [ ]:
if RUN_TRAINING:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_notebook_exploration")
    run_dir = PROJECT_ROOT / "results" / "runs" / run_id
    trainer = PokemonAITrainer.from_rom(ROM_PATH, config, run_dir)
    metrics = trainer.train()
    print(metrics)
else:
    print("Entrenamiento desactivado (RUN_TRAINING=False).")

## 6. Resultados históricos y alcance

La única salida textual conservada del notebook original contiene tres episodios,
todos truncados en 5 000 pasos: recompensas `3779.74`, `4457.80` y `4348.40`
(media `4195.31`). No se reprodujeron durante esta reorganización y no son
comparables directamente con el pipeline corregido.

![Recompensas históricas](../results/figures/legacy_rewards.png)

No existe evidencia suficiente para afirmar convergencia o una política de juego
estable. Consulta [`docs/results.md`](../docs/results.md) para las limitaciones y
[`docs/methodology.md`](../docs/methodology.md) para las correcciones técnicas.